# 🏭 PumpGuard AI — IoT Demo trên Google Colab

> Chạy toàn bộ hệ thống IoT (MQTT + FastAPI backend + Dashboard) trực tiếp trên Colab, không cần cài gì trên máy tính.

**Các bước:**
1. 🔑 Điền API key (Gemini free)
2. 📦 Cài dependencies
3. 🦟 Khởi động MQTT broker
4. 🚀 Khởi động backend
5. 🌐 Lấy public URL qua ngrok
6. 📊 Mở dashboard!

---
⏱️ **Thời gian cài đặt:** ~2-3 phút lần đầu

## Bước 1 — 🔑 Cấu hình API Keys

Điền key vào đây trước khi chạy các bước tiếp theo.

In [ ]:
# ═══════════════════════════════════════════════════════
# ĐIỀN API KEY CỦA BẠN VÀO ĐÂY
# ═══════════════════════════════════════════════════════

# Google Gemini — Lấy miễn phí tại: https://aistudio.google.com/apikey
GEMINI_API_KEY = "AIzaSy-xxxx-thay-bang-key-cua-ban"

# ngrok — Lấy free authtoken tại: https://dashboard.ngrok.com/authtokens
NGROK_TOKEN = "xxxx-thay-bang-ngrok-token-cua-ban"

# Email alert (tuỳ chọn — để trống nếu không cần)
RESEND_API_KEY = ""
ALERT_TO = ""

print('✅ Config đã được set. Tiếp tục bước tiếp theo.')

## Bước 2 — 📦 Clone repo & cài dependencies

In [ ]:
import os, subprocess, sys

# Clone repo
if not os.path.exists('/content/demo-iot'):
    print('📥 Cloning repo...')
    subprocess.run(['git', 'clone', 'https://github.com/hoamx2602/demo-iot.git',
                    '/content/demo-iot', '--branch', 'cloud-classroom', '-q'], check=True)
    print('✅ Clone xong!')
else:
    print('✅ Repo đã tồn tại, bỏ qua clone.')

os.chdir('/content/demo-iot')

# Cài Python dependencies
print('📦 Cài Python packages...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'backend/requirements.txt'], check=True)

# Cài ngrok
print('🌐 Cài pyngrok...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyngrok'], check=True)

print('\n✅ Tất cả dependencies đã được cài!')

## Bước 3 — 🦟 Khởi động Mosquitto MQTT Broker

In [ ]:
import subprocess, time

# Cài mosquitto
print('🦟 Cài Mosquitto...')
subprocess.run(['apt-get', 'install', '-y', '-q', 'mosquitto'], check=True,
               capture_output=True)

# Tạo config cho mosquitto (allow anonymous)
with open('/tmp/mosquitto.conf', 'w') as f:
    f.write('listener 1883\nallow_anonymous true\n')

# Khởi động mosquitto
proc = subprocess.Popen(['mosquitto', '-c', '/tmp/mosquitto.conf'],
                         stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(1)

if proc.poll() is None:
    print(f'✅ Mosquitto đang chạy (PID {proc.pid}) trên port 1883')
else:
    print('❌ Mosquitto không khởi động được. Xem lỗi bên dưới:')
    subprocess.run(['mosquitto', '-c', '/tmp/mosquitto.conf', '--verbose'])

## Bước 4 — ✍️ Tạo file .env

In [ ]:
import os

os.chdir('/content/demo-iot')

env_content = f"""AI_PROVIDER=gemini
GEMINI_API_KEY={GEMINI_API_KEY}
MQTT_HOST=localhost
MQTT_PORT=1883
RESEND_API_KEY={RESEND_API_KEY}
ALERT_TO={ALERT_TO}
"""

with open('backend/.env', 'w') as f:
    f.write(env_content)

print('✅ .env đã được tạo!')
print(f'   AI_PROVIDER  = gemini')
print(f'   GEMINI_KEY   = {GEMINI_API_KEY[:12]}...' if len(GEMINI_API_KEY) > 12 else '   ⚠️ Gemini key chưa được điền!')
print(f'   MQTT_HOST    = localhost:1883')

## Bước 5 — 🚀 Khởi động FastAPI Backend

In [ ]:
import subprocess, time, requests, os

os.chdir('/content/demo-iot')

# Kill process cũ nếu có
subprocess.run(['pkill', '-f', 'uvicorn'], capture_output=True)
time.sleep(1)

# Start backend
log_file = open('/tmp/backend.log', 'w')
proc = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'backend.server:app',
     '--host', '0.0.0.0', '--port', '8000'],
    stdout=log_file, stderr=log_file,
    env={**os.environ, **dict(
        line.split('=', 1) for line in open('backend/.env').read().strip().splitlines()
        if '=' in line and not line.startswith('#')
    )}
)

# Đợi backend lên
print('⏳ Đợi backend khởi động...')
for i in range(15):
    time.sleep(1)
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            data = r.json()
            print(f'\n✅ Backend đang chạy! (PID {proc.pid})')
            print(f'   AI provider: {data.get("ai_provider", "?")} | API key: {"✅ OK" if data.get("api_key_configured") else "❌ Chưa set"}')
            break
    except:
        print('.', end='', flush=True)
else:
    print('\n❌ Backend không lên được. Xem log:')
    with open('/tmp/backend.log') as f:
        print(f.read()[-2000:])

## Bước 6 — 🌐 Tạo Public URL với ngrok

> **Cần ngrok free account:** [dashboard.ngrok.com/authtokens](https://dashboard.ngrok.com/authtokens)

In [ ]:
from pyngrok import ngrok, conf
import time

# Config ngrok token
conf.get_default().auth_token = NGROK_TOKEN

# Kill tunnel cũ
ngrok.kill()
time.sleep(1)

# Tạo tunnel HTTP → port 8000
tunnel = ngrok.connect(8000, bind_tls=True)
public_url = tunnel.public_url

print('\n' + '='*60)
print('🎉 PUMPGUARD AI ĐANG CHẠY!')
print('='*60)
print(f'\n🌐 Dashboard URL (share cho học viên):')
print(f'   {public_url}/dashboard/')
print(f'\n🔗 API Docs:')
print(f'   {public_url}/docs')
print(f'\n❤️  Health check:')
print(f'   {public_url}/health')
print('\n' + '='*60)
print('⚠️  URL này có hiệu lực trong 2 giờ (ngrok free tier)')
print('   Chạy lại cell này để lấy URL mới nếu hết hạn')
print('='*60)

## Bước 7 (Tuỳ chọn) — 📊 Chạy MQTT Data Replay

Chạy cell này để stream dữ liệu sensor thật từ CSV lên dashboard.

> ⚠️ Cell này sẽ **chạy liên tục** (blocking). Dừng bằng nút ⏹ trên Colab.

In [ ]:
import os
os.chdir('/content/demo-iot')

# Kiểm tra file data
if not os.path.exists('data/sensor.csv'):
    print('⚠️  Không tìm thấy data/sensor.csv')
    print('   Dùng Operator Controls trên dashboard thay thế:')
    print('   ▶ Normal Operation | ⚠ Simulate Anomaly | 🔴 Simulate Critical')
else:
    print('▶ Bắt đầu replay dữ liệu sensor...')
    print('  Nhấn ⏹ để dừng')
    print('-' * 50)
    # start-at-anomaly: bắt đầu gần điểm anomaly để demo nhanh hơn
    !python scripts/mqtt_replay.py \
        --csv data/sensor.csv \
        --config data/sensor_groups.json \
        --start-at-anomaly \
        --compression 360

---
## 🛠 Troubleshooting

| Vấn đề | Giải pháp |
|--------|----------|
| Dashboard không kết nối WebSocket | Chạy lại Bước 6 để lấy URL mới |
| AI chỉ hiện MOCK data | Kiểm tra GEMINI_API_KEY ở Bước 1 |
| ngrok tunnel hết hạn | Chạy lại cell Bước 6 |
| Colab Runtime ngắt | Chạy lại từ Bước 3 (skip Bước 2) |
| Backend không khởi động | Xem log: `!cat /tmp/backend.log` |

### Xem log backend
```python
!tail -30 /tmp/backend.log
```

### Khởi động lại toàn bộ
Chạy lại từ **Bước 3** (không cần chạy Bước 2 lại).